# 12 Evaluation
In this notebook we evaluate the recommender models built on different features (see Notebooks 08 to 10 for more information). We use parent genres, which we built and predicted in Notebook 11, to evaluate how good a recommendation is based on a given song.

Model A and Model B are evaluated here. Model C (lyrics text) is out of scope for now.

Each model is evaluated sequentially: build its validation features, score it, then delete its intermediates before moving to the next model. This keeps peak memory low, since we never hold more than one model's worth of merged data at a time. It also keeps each model's `genre_lists` correctly aligned with its own validation rows, since inner joins can drop different tracks for each model (e.g. tracks without lyrics features won't survive Model B's merge but we trained Model B on songs with lyrics features only too).

In [3]:
import joblib
import random
import gc
import psutil
from pathlib import Path
import sys
import ast
import numpy as np
import pandas as pd
import warnings
from collections import Counter

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

processed_dir = PROJECT_ROOT / "data" / "processed"

def check_memory():
    print(f"{psutil.virtual_memory().percent}% used | {psutil.virtual_memory().available / 1e9:.2f} GB available")

check_memory()

89.3% used | 0.84 GB available


## Shared helpers

These are used for every model, so we define them once up front: the genre-list parser, the Jaccard metric, and the two baselines.

In [4]:
def to_list(val):
    """Normalize a genre cell into a plain python list, handling lists/arrays/NaN/stringified lists."""
    if isinstance(val, (list, set, np.ndarray)):
        return list(val)
    if pd.isna(val):
        return []
    return ast.literal_eval(val)

In [5]:
def jaccard(set_a, set_b):
    set_a, set_b = set(set_a), set(set_b)
    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

def genre_overlap_at_k(model, X_scaled, genre_lists, k=10):
    _, indices = model.kneighbors(X_scaled, n_neighbors=k + 1)  # +1 to drop self-match
    scores = []
    for i, neighbors in enumerate(indices):
        query_genres = genre_lists[i]
        neighbor_idxs = [j for j in neighbors if j != i][:k]
        overlap = [jaccard(query_genres, genre_lists[j]) for j in neighbor_idxs]
        scores.append(sum(overlap) / len(overlap))
    return sum(scores) / len(scores)

In [6]:
k_values = [5, 10]
results_rows = {f"k={k}": {} for k in k_values}

## Model A: audio features only

Load the validation set, merge with audio features, build the candidate feature list, scale with the saved `scaler_a`, and score with `model_a`.

In [7]:
df = pd.read_parquet(processed_dir / "tracks_with_predicted_genres.parquet")
audio_features = pd.read_parquet(processed_dir / "audio_features_clean.parquet")

model_a = joblib.load("models/model_a.joblib")
scaler_a = joblib.load("models/scaler_a.joblib")

check_memory()

98.9% used | 0.09 GB available


In [8]:
audio_tracks = df[
    df["id"].isin(audio_features["track_id"])
].copy()

audio_master = audio_tracks.merge(
    audio_features,
    left_on="id",
    right_on="track_id",
    how="inner"
).drop(columns=["lyrics"], errors="ignore")

exclude_columns = [
    "id",
    "track_id",
    "name",
    "lyrics",
    "analysis_url",
    "track_href",
    "uri",
    "href",
    "preview_url",
    "available_markets",
    "playlist",
    "track_name_prev",
    "artists_id",
    "album_id",
    "country",
    "genre_list",
    "genre_canonical",
    "predicted_genre_list",
]

candidate_features = [
    col
    for col in audio_master.columns
    if col not in exclude_columns
]

# audio_tracks and audio_features are now folded into audio_master, delete rest
del audio_tracks, audio_features
gc.collect()
check_memory()

88.0% used | 0.95 GB available


In [9]:
numeric_features = audio_master[
    candidate_features
].select_dtypes(
    include=np.number
).columns.tolist()

variance_threshold = 0.001

feature_variance = audio_master[numeric_features].var()

low_variance_features = (
    feature_variance[feature_variance < variance_threshold]
    .index
    .tolist()
)

model_a_features = [
    feature
    for feature in numeric_features
    if feature not in low_variance_features
]

X_audio = audio_master[model_a_features].copy()

In [10]:
# Genre lists for Model A 
genre_lists_a = audio_master["predicted_genre_list"].apply(to_list).tolist()

X_a_scaled = scaler_a.transform(X_audio)

for k in k_values:
    results_rows[f"k={k}"]["Model A"] = genre_overlap_at_k(model_a, X_a_scaled, genre_lists_a, k=k)

results_rows

{'k=5': {'Model A': 0.21432618842481474},
 'k=10': {'Model A': 0.21452675010471944}}

In [ ]:
# Done with Model A, delete unnecessary stuff
del audio_master, X_audio, X_a_scaled, model_a, scaler_a
gc.collect()
check_memory()

85.4% used | 1.16 GB available


## Model B:  audio features + lyrics features

Rebuild from `df`, merge in lyrics features this time, scale with the saved `scaler_b` and score with `model_b`. Lyrics features consist of word/sentence counts, syllables per word, sentence similarity and vocabulary wealth.

In [12]:
lyrics_features = pd.read_parquet(processed_dir / "lyrics_features_valid_clean.parquet")
audio_features = pd.read_parquet(processed_dir / "audio_features_clean.parquet")

model_b = joblib.load("models/model_b.joblib")
scaler_b = joblib.load("models/scaler_b.joblib")

audio_tracks = df[
    df["id"].isin(audio_features["track_id"])
].copy()

audio_master = audio_tracks.merge(
    audio_features,
    left_on="id",
    right_on="track_id",
    how="inner"
).drop(columns=["lyrics"], errors="ignore")

del audio_tracks, audio_features
gc.collect()
check_memory()

86.6% used | 1.06 GB available


In [ ]:
model_b_master = audio_master.merge(
    lyrics_features,
    left_on="id",
    right_on="track_id",
    how="inner",
    suffixes=("", "_lyrics")
)

lyrics_feature_columns = [
    col
    for col in lyrics_features.columns
    if col != "track_id"
]

model_b_features = model_a_features + lyrics_feature_columns

# delete again after using
del audio_master, lyrics_features
gc.collect()
check_memory()

85.2% used | 1.17 GB available


In [14]:
X_model_b = model_b_master[model_b_features].copy()

X_b_scaled = scaler_b.transform(X_model_b)

genre_lists_b = model_b_master["predicted_genre_list"].apply(to_list).tolist()

for k in k_values:
    results_rows[f"k={k}"]["Model B"] = genre_overlap_at_k(model_b, X_b_scaled, genre_lists_b, k=k)

results_rows

{'k=5': {'Model A': 0.21432618842481474, 'Model B': 0.25109947672576716},
 'k=10': {'Model A': 0.21452675010471944, 'Model B': 0.25138976122681234}}

In [ ]:
del model_b_master, X_model_b, X_b_scaled, model_b, scaler_b
gc.collect()
check_memory()

66.9% used | 2.62 GB available
